# 🧪 Sesión 17 · Control de Acceso Basado en Roles (RBAC)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué diferencia hay entre no estar autenticado y no tener permiso?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/admin/panel   Authorization: Bearer <token_admin>
HTTP/1.1 200 OK
{"panel":"admin"}

GET /api/admin/panel   Authorization: Bearer <token_user>
HTTP/1.1 403 Forbidden

GET /api/admin/panel   (sin token)
HTTP/1.1 401 Unauthorized

GET /api/perfil        Authorization: Bearer <token_user>
HTTP/1.1 200 OK
{"usuario":"ana","rol":"user"}
```

_El 403 no es un error de credenciales: es una decisión de autorización._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Incluir el rol dentro de los claims del token.
- Distinguir 401 (no autenticado) de 403 (sin permiso).
- Proteger una ruta administrativa con RBAC.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El administrador obtiene su token → HTTP 200
- [ ] El usuario normal obtiene su token → HTTP 200
- [ ] El admin entra al panel → HTTP 200
- [ ] El usuario normal recibe 403 → HTTP 403
- [ ] Sin token recibe 401 → HTTP 401
- [ ] El perfil funciona para el usuario normal → HTTP 200

## 🧩 El reto

Implementa autenticación con roles: un usuario normal y un administrador. El panel administrativo solo debe abrirse para admin.

**Requisitos del contrato**

- Usuarios de prueba: admin/admin123 (rol admin) y ana/secreta (rol user).
- El token incluye el rol del usuario en sus claims.
- GET /api/admin/panel responde 200 solo para rol admin.
- Un usuario autenticado con rol user recibe 403.
- Sin token se responde 401, no 403.
- GET /api/perfil funciona para cualquier usuario autenticado.

> **Pista conceptual:** Primero autentica (¿quién eres?) y después autoriza (¿tu rol alcanza?). Si devuelves 403 a quien no tiene token, estás mezclando las dos preguntas.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion17.md` con la matriz rol → permiso.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tus respuestas de autorización se verán así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El usuario normal obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El admin entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200,
  'json_contiene': {'panel': 'admin'}},
 {'descripcion': 'El usuario normal recibe 403',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'Sin token recibe 401', 'metodo': 'GET', 'ruta': '/api/admin/panel', 'estado': 401},
 {'descripcion': 'El perfil funciona para el usuario normal',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'rol': 'user'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion17.md` a `auditorias/auditoria_sesion17.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
